In [1]:
!pip install torchvision

In [2]:
import torch
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [3]:
# Image load => transform => dataset of all imgs
class ImageProcessor:
    def __init__(self, root_dir_path, transformations):
        self.root_dir_path = root_dir_path
        self.transformations = transformations

        # list of path for all images
        self.all_imgs_path = [os.path.join(root_dir_path, img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_imgs_path)

    def __getItem__(self, idx):
        img_path = self.all_imgs_path[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)
        return img
        
        
    

In [4]:
root_dir_path = "./img_align_celeba"
transformations = transforms.Compose([
    transforms.CenterCrop(178), # 178*218 => 178*178
    transforms.Resize(64), # 64*64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # [-1, 1]
])

In [5]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [6]:
dataloader = DataLoader(dataset, batch_size = 128, shuffle = True)
print(type(dataloader))

<class 'torch.utils.data.dataloader.DataLoader'>


# Generator Network

In [8]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [12]:
class Generator(nn.Module): # Generator gets fake data which we represent here by z which is basically the random vecotrs of noise(fake imgs data)
    def __init__(self, z_dim = 100, img_channels = 3): # 3 is for RGB, z is noise the image random vectors
        super(Generator, self).__init__()

        # Fully connected layer

        self.model = nn.Sequential(
            nn.Linear(z_dim, 256), # 100 => 256
            nn.ReLU(),


            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, 64*64*img_channels),
            nn.Tanh() # [-1, 1] normalize pixel in this range

        )

    def forward(self, z):
        img = self.model(z)
        img = img.view(img.size(0), 3, 64, 64)
        return img


        # fake image = 64 * 64 * 3 * batchSize so thats's 4D

# Discriminator Network

In [13]:
class Discriminator(nn.Module): 
    def __init__(self, img_channels = 3): # 3 is for RGB,
        super(Discriminator, self).__init__()

        # Fully connected layer

        self.model = nn.Sequential(

            nn.Flatten(), # 4D Tensors => 1D
                
            nn.Linear(img_channels, 1024), # 100 => 256
            nn.LeakyReLU(),


            nn.Linear(1024, 512),
            nn.LeakyReLU(),

            nn.Linear(512, 256),
            nn.LeakyReLU(),

            nn.Linear(256, 1),
            nn.Sigmoid() # Probability of being fake or real

        )

    def forward(self, z):
        return self.model(img)


In [16]:
GAN_loss = nn.BCELoss() # Binary Cross Entropy loss

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr = 0.0002, betas = (0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr = 0.0002, betas = (0.5, 0.999))



# Device

In [17]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("My device: ", device)

My device:  mps


In [18]:
generator = generator.to(device)
discriminator = discriminator.to(device)

# Training the GAN

In [23]:
def trian(generator, discriminator, dataloader, epochs = 10):

    for epoch in range(epochs):
        for i, img in enumerate(dataloader):
            real_imgs = img.to(device)
            batch_size =  real_imgs.size(0)

            # create real img labels and fake image labels
            real_labels = torch.ones(batch_size, 1).to(device) # [1,1,1,.....]
            fake_labels = torch.zeros(batch_size, 1).to(deivce) # [0,0,0,......]

            # Trian the disscriminator
            d_optimizer.zero_grad()

            noise = torch.randn(batch_size, 100).to(device)
            fake_imgs = generator(noise)

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            d_optimizer.step()

            # Train the generator
            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_img),  real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch in {epoch + 1}/{epochs}... batch: {i + 1}... G-loss: {g_loss.item()} & D-loss: {d_loss.item()}")
        # save generated img for each epoch
        save_generated_images(generator, epoch, device)
            

In [24]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, num_imgs = 8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_gird(generated_imgs, nrow = 4, Normalize = True)

    plt.imhshow(np.transpose(grid, 1, 2, 0))
    plt.title(f"epoch {epoch + 1}")
    plt.axis("off")
    plt.show()

In [25]:
trian(generator, discriminator, dataloader, epochs = 10)

TypeError: 'ImageProcessor' object is not subscriptable